# Revisión de integridad de GeoTIFF NDVI bisemanal

1. **Objetivo del notebook**
- Revisar la integridad de los GeoTIFF NDVI externos y detectar archivos faltantes, corruptos o incompletos.

2. **Decisiones metodológicas tomadas en la construcción del notebook**
- Usar la ruta canónica externa definida en src.paths.py.
- Leer por bloques para validar integridad sin redescargar ni mover GeoTIFF.

3. **Archivos generados en el notebook y ubicación de guardado**
- CSV de resultados de integridad en data/diagnostics.
- CSV de archivos pendientes de revisión/redescarga en data/diagnostics.

Cada sección Markdown posterior explica brevemente qué hace el código que la sigue.


Este notebook revisa los GeoTIFF pendientes para `minimo_bisemanal` y `desvest_bisemanal`. Abre cada archivo con Rasterio y fuerza la lectura por bloques para detectar archivos que existen, pero están corruptos o incompletos.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.paths import *
ensure_directories()


In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import rasterio
from rasterio.errors import RasterioIOError


BASE_DIR = require_ndvi_external_dir()
OUTPUT_DIR = DATA_DIAGNOSTICS

EXPECTED_BIWEEKS = range(1, 27)
PENDING_CONFIG = {
    "ndvi_minimo_bisemanal": {
        "folder": "minimo_bisemanal",
        "suffix": "min",
    },
    "ndvi_desvest_bisemanal": {
        "folder": "desvest_bisemanal",
        "suffix": "std",
    },
}

PROBLEM_STATUSES = {"MISSING", "DUPLICATE_FILE", "DAMAGED", "SUSPICIOUS_EMPTY"}

In [3]:
def biweek_from_filename(path: Path) -> int | None:
    match = re.search(r"_BI(\d{2})_", path.name)
    if not match:
        return None
    return int(match.group(1))


def build_expected_inventory() -> pd.DataFrame:
    rows = []
    for column, config in PENDING_CONFIG.items():
        folder = config["folder"]
        suffix = config["suffix"]
        stat_dir = BASE_DIR / folder

        for biweek in EXPECTED_BIWEEKS:
            matches = sorted(stat_dir.glob(f"*_BI{biweek:02d}_{suffix}.tif")) if stat_dir.exists() else []
            if not matches:
                rows.append({
                    "column": column,
                    "folder": folder,
                    "suffix": suffix,
                    "biweek": biweek,
                    "path": stat_dir / f"MISSING_BI{biweek:02d}_{suffix}.tif",
                    "inventory_status": "MISSING",
                    "match_count": 0,
                })
            elif len(matches) > 1:
                for path in matches:
                    rows.append({
                        "column": column,
                        "folder": folder,
                        "suffix": suffix,
                        "biweek": biweek,
                        "path": path,
                        "inventory_status": "DUPLICATE_FILE",
                        "match_count": len(matches),
                    })
            else:
                rows.append({
                    "column": column,
                    "folder": folder,
                    "suffix": suffix,
                    "biweek": biweek,
                    "path": matches[0],
                    "inventory_status": "FOUND",
                    "match_count": 1,
                })

    inventory = pd.DataFrame(rows)
    inventory["path"] = inventory["path"].map(str)
    return inventory


inventory = build_expected_inventory()
inventory.groupby(["folder", "inventory_status"]).size().unstack(fill_value=0)

inventory_status,FOUND
folder,
desvest_bisemanal,26
minimo_bisemanal,26


In [4]:
def check_tiff_integrity(path: str) -> dict:
    result = {
        "status": "OK",
        "error_type": "",
        "error_message": "",
        "driver": "",
        "width": np.nan,
        "height": np.nan,
        "count": np.nan,
        "dtype": "",
        "crs": "",
        "nodata": np.nan,
        "finite_pixels": 0,
        "min_value": np.nan,
        "max_value": np.nan,
    }

    path_obj = Path(path)
    if not path_obj.exists():
        result.update(status="MISSING", error_message="El archivo no existe")
        return result

    try:
        with rasterio.open(path_obj) as src:
            result.update({
                "driver": src.driver,
                "width": src.width,
                "height": src.height,
                "count": src.count,
                "dtype": src.dtypes[0] if src.dtypes else "",
                "crs": str(src.crs),
                "nodata": src.nodata,
            })

            if src.count < 1 or src.width <= 0 or src.height <= 0:
                result.update(status="DAMAGED", error_message="Dimensiones o bandas invalidas")
                return result

            finite_pixels = 0
            min_value = np.inf
            max_value = -np.inf

            for _, window in src.block_windows(1):
                block = src.read(1, window=window, masked=True)
                values = np.asarray(block.compressed(), dtype="float64")
                values = values[np.isfinite(values)]
                if values.size:
                    finite_pixels += int(values.size)
                    min_value = min(min_value, float(values.min()))
                    max_value = max(max_value, float(values.max()))

            result["finite_pixels"] = finite_pixels
            if finite_pixels:
                result["min_value"] = min_value
                result["max_value"] = max_value
            else:
                result.update(status="SUSPICIOUS_EMPTY", error_message="No hay pixeles finitos validos")

    except Exception as exc:
        result.update(
            status="DAMAGED",
            error_type=type(exc).__name__,
            error_message=str(exc),
        )

    return result

In [5]:
rows = []
for item in inventory.itertuples(index=False):
    base = item._asdict()
    if item.inventory_status == "FOUND":
        base.update(check_tiff_integrity(item.path))
    else:
        base.update({
            "status": item.inventory_status,
            "error_type": "",
            "error_message": item.inventory_status,
            "driver": "",
            "width": np.nan,
            "height": np.nan,
            "count": np.nan,
            "dtype": "",
            "crs": "",
            "nodata": np.nan,
            "finite_pixels": 0,
            "min_value": np.nan,
            "max_value": np.nan,
        })
    rows.append(base)

results = pd.DataFrame(rows)
summary = results.groupby(["folder", "status"]).size().unstack(fill_value=0)
summary

status,OK
folder,
desvest_bisemanal,26
minimo_bisemanal,26


In [6]:
problem_files = results[results["status"].isin(PROBLEM_STATUSES)].copy()
problem_files = problem_files.sort_values(["folder", "biweek", "path"])
problem_files[["folder", "biweek", "status", "error_type", "error_message", "path"]]

,folder,biweek,status,error_type,error_message,path


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_path = OUTPUT_DIR / "ndvi_pending_tiff_integrity_results.csv"
problems_path = OUTPUT_DIR / "ndvi_pending_tiff_files_to_redownload.csv"

results.to_csv(results_path, index=False, encoding="utf-8-sig")
problem_files.to_csv(problems_path, index=False, encoding="utf-8-sig")

print(f"Resultados completos: {results_path}")
print(f"Archivos para revisar/redescargar: {problems_path}")
print(f"Total con problemas: {len(problem_files)}")

Resultados completos: C:\tesis_aedes_grilla_2024\data\diagnostics\ndvi_pending_tiff_integrity_results.csv
Archivos para revisar/redescargar: C:\tesis_aedes_grilla_2024\data\diagnostics\ndvi_pending_tiff_files_to_redownload.csv
Total con problemas: 0


In [8]:
print("Path migration check completed.")


Path migration check completed.
